# Model & Quality Monitors Plus Dashboard & Reports

This code adds Data & Model Quality Monitors in the AWS environment to track shifts or degredation in the data or model. The code here follows the following steps: 

1. Environment Setup
2. Endpoint Deployment with Data Capture Enabled
3. Data Monitor Setup, plus Monitoring Schedule
4. Model Monitor Setup, plus Monitoring Schedule
5. Executing Models to see what production data will be like
6. Infrastructure Monitors and Alarms
7. Dashboard
8. Verification code, to ensure monitors are workings as expected
9. Monitor Report Download
10. Cleanup

In a production environment, where data is not static, it is critical to monitor different systems in order to ensure proper functionality. By monitoring the infrastructure, as well as the data, and the model itself, we are able to track performance and alert the appropriate team if anything isn't operating as intended. 

Attribution: This code was made with the help of AWS tutorials, reference of lab resources in AAI 540, as well as both Claude Code and Perplexity accessed February, 2026.

## 1. Environment Setup

In [1]:
#1.1 Library Imports
import sagemaker
from sagemaker import Session
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker import image_uris, get_execution_role
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput,
)

import s3fs
import boto3
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta, timezone
from io import StringIO
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#1.2 Configuration

region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

## 2. Deploy Endpoint with Data Capture for Model Monitor

In [3]:
#2.1 Locate model artifacts
local_base = Path("/tmp/Models/benchmarks")
s3_base = f"s3://{bucket}/models/benchmarks"

xgb_paths = {
    "local_tar.gz": local_base / "xgboost/model.tar.gz",
    "s3_tar.gz": f"{s3_base}/xgboost/model.tar.gz",
}

model_data = (
    str(xgb_paths["local_tar.gz"])
    if xgb_paths["local_tar.gz"].exists()
    else xgb_paths["s3_tar.gz"]
)

In [4]:
#2.2 Container
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
)

new_xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

In [5]:
#2.3 Model & Endpoint
xg_model = Model(
    image_uri=xgboost_container,
    model_data=model_data,
    role=role,
    sagemaker_session=session,
)

data_capture_prefix = f"{prefix}/datacapture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

# Deploy with data capture enabled
xg_predictor = xg_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=new_xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print("Endpoint:", new_xgb_endpoint_name)

------!Endpoint: xgb-benchmark-endpoint-20260211-163922


In [6]:
#2.4 Wait until endpoint is InService

print("Waiting for endpoint to be ready...")
while True:
    resp = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
    status = resp["EndpointStatus"]
    print(" Status:", status)
    if status == "InService":
        print("Endpoint is ready!")
        break
    if status == "Failed":
        raise RuntimeError(f"Endpoint deployment failed: {resp.get('FailureReason')}")
    time.sleep(30)

Waiting for endpoint to be ready...
 Status: InService
Endpoint is ready!


## 3. DQ Data Monitor
Data Quality

In [7]:
# 3.1: Create Data Quality Monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

In [8]:
# 3.2: Generate baseline using your data (creates stats/constraints)
baseline_dataset_uri = "s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv"
dq_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/dq-baseline"

# Run baselining job
dq_baseline_job = dq_monitor.suggest_baseline(
    baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=dq_baseline_uri,
    wait=True,
    logs=False,
)

print("✅ DQ baseline created!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-11-16-42-55-429


...........................................................!✅ DQ baseline created!


In [9]:
# 3.3: Get the generated stats/constraints URIs
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
except:
    # Fallback: find files manually
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/dq-baseline/")
    for obj in resp.get("Contents", []):
        if "statistics.json" in obj["Key"]:
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if "constraints.json" in obj["Key"]:
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"

print("Data Quality Stats:", dq_stats_uri)
print("Data Quality Constraints:", dq_constraints_uri)


Data Quality Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/dq-baseline/statistics.json
Data Quality Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/dq-baseline/constraints.json


In [10]:
# 3.4: Create monitoring schedule using new baseline
schedule_name_xgb_dq = "xgb-data-quality-schedule"

try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
except:
    pass  # No existing schedule

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=new_xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics=dq_stats_uri,
    constraints=dq_constraints_uri,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("✅ Data quality schedule live:", schedule_name_xgb_dq)


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule


✅ Data quality schedule live: xgb-data-quality-schedule


In [11]:
dq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-data-quality-schedule',
 'MonitoringScheduleName': 'xgb-data-quality-schedule',
 'MonitoringScheduleStatus': 'Pending',
 'MonitoringType': 'DataQuality',
 'CreationTime': datetime.datetime(2026, 2, 11, 16, 47, 59, 668000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 11, 16, 47, 59, 732000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'data-quality-job-definition-2026-02-11-16-47-58-603',
  'MonitoringType': 'DataQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260211-163922',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-data-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 11, 7, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 11, 7, 2, 3, 526000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(2026, 2, 

In [12]:
dq_executions = dq_monitor.list_executions()
dq_executions

## 4. MQ Model Monitor 
Model Quality

In [13]:
#4.1 Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [14]:
#4.2 Create a model quality baseline dataset

# Load data
baseline_df = pd.read_csv(f"s3://{bucket}/{prefix}/baseline_normalized.csv")
X_baseline = baseline_df.drop("target", axis=1)

# Get predictions from endpoint
csv_payload = X_baseline.to_csv(header=False, index=False)
predictor = Predictor(endpoint_name=new_xgb_endpoint_name, sagemaker_session=session)
response = predictor.predict(
    data=csv_payload,
    initial_args={"ContentType": "text/csv", "Accept": "text/csv"},
)

# Parse multi-class predictions
predictions_text = response.decode("utf-8").strip().split("\n")
prob_matrix = np.array([[float(x) for x in line.split(",") if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)  # argmax for multiclass

# Create baseline dataset
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

mq_baseline_key = f"{prefix}/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False)
s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue(),
    ContentType="text/csv",
)

mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print("✅ MQ baseline uploaded:", mq_baseline_uri)

✅ MQ baseline uploaded: s3://sagemaker-us-east-1-418418308994/models/benchmarks/mq_baseline.csv


In [15]:
#4.2 Run baselining job
mq_baseline_folder = f"s3://{bucket}/{prefix}/mq-baseline"

mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/mq-baseline-simple",
    problem_type="MulticlassClassification",
    inference_attribute="prediction",
    ground_truth_attribute="ground_truth_label",
    wait=True,
    logs=True,
)

print("✅ Baselining job complete!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-11-16-48-03-979


................2026-02-11 16:50:48.974665: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-11 16:50:48.974703: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-02-11 16:50:50.382833: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-02-11 16:50:50.382867: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2026-02-11 16:50:50.382890: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (ip-10-2-212-152.ec2.internal): /proc/driver/nvidia/version does not exist
2026-02-11 16:50:50.383164: I te

In [16]:
#4.3 Get baseline files
job_desc = mq_monitor.latest_baselining_job.describe()
output_uri = job_desc['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri']
mq_stats_uri = f"{output_uri}/statistics.json"
mq_constraints_uri = f"{output_uri}/constraints.json"
print("✅ Stats:", mq_stats_uri)
print("✅ Constraints:", mq_constraints_uri)

✅ Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline-simple/statistics.json
✅ Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline-simple/constraints.json


In [17]:
# Diagnose baselining failure
job_desc = mq_monitor.latest_baselining_job.describe()
print("Job Status:", job_desc["ProcessingJobStatus"])
print("Failure Reason:", job_desc.get("FailureReason", "None"))
print("Exit Message:", job_desc.get("ExitMessage", "None"))


Job Status: Completed
Failure Reason: None
Exit Message: Completed: Job completed successfully with no violations.


In [18]:
#4.4 Create a Model Quality Monitoring Schedule

mq_schedule_name = "xgb-model-quality-schedule"

mq_monitor.create_monitoring_schedule(
    monitor_schedule_name=mq_schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=new_xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",
    ),
    problem_type="MulticlassClassification",
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Model quality schedule:", mq_schedule_name)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


Model quality schedule: xgb-model-quality-schedule


In [19]:
#4.5 Create a Ground Truth File for Current Hour

s3_resource = boto3.resource("s3")

target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)

# Upload to path
gt_key = f"{prefix}/ground-truth/{target_hour.strftime('%Y%m%d%H')}/ground-truth.jsonl"
records = [json.dumps({
    "groundTruthData": {"data": str(int(label)), "encoding": "CSV"},
    "eventMetadata": {"eventId": f"gt-{i}", "inferenceTime": target_hour.isoformat()},
    "eventVersion": "0"
}) for i, label in enumerate(baseline_df['target'].iloc[:200])]

s3.put_object(Bucket=bucket, Key=gt_key, Body="\n".join(records))

print(f"✅ Ground truth uploaded to: s3://{bucket}/{gt_key}")

Ground truth uploaded to: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/11/16/ground-truth.jsonl


/tmp/ipykernel_370/367159885.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  target_hour = datetime.utcnow().replace(minute=0, second=0, microsecond=0)


In [55]:
#4.6 Send predictions
test_data = baseline_df.drop("target", axis=1).iloc[:100]
predictor.predict(test_data.to_csv(header=False, index=False), 
                  initial_args={'ContentType': 'text/csv'})
print("✅ Predictions + ground truth ready!")

✅ Predictions + ground truth ready!


In [56]:
#4.8 Check monitor and executions
mq_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:418418308994:monitoring-schedule/xgb-model-quality-schedule',
 'MonitoringScheduleName': 'xgb-model-quality-schedule',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2026, 2, 11, 16, 53, 30, 66000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 11, 18, 13, 8, 729000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2026-02-11-16-53-29-285',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'xgb-benchmark-endpoint-20260211-163922',
 'LastMonitoringExecutionSummary': {'MonitoringScheduleName': 'xgb-model-quality-schedule',
  'ScheduledTime': datetime.datetime(2026, 2, 11, 18, 0, tzinfo=tzlocal()),
  'CreationTime': datetime.datetime(2026, 2, 11, 18, 2, 42, 315000, tzinfo=tzlocal()),
  'LastModifiedTime': datetime.datetime(

In [57]:
mq_executions = mq_monitor.list_executions()
mq_executions

## 5. Trigger Executions Manually

What happens during runs: 

1. DataCapture/*.jsonl    → Monitor input
2. ground-truth/*.jsonl   → GT input  
3. Merge job              → Joins predictions+GT
4. Quality analysis       → Computes F1/accuracy
5. S3 reports + CloudWatch → f1=0.94, auc=0.93 🎉
   

In [67]:
#5.1 Confirm Data Flow
now_hh = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0).strftime('%Y%m%d%H')

print(f"Checking hour {now_hh} UTC...")

dc_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{now_hh}"
gt_path = f"{bucket}/{prefix}/ground-truth/{now_hh}"

dc_ok = False
gt_ok = False

try:
    dc_files = fs.ls(dc_path, detail=True)
    print(f"✅ DataCapture: {len(dc_files)} files")
    dc_ok = True
except:
    print("⏳ DataCapture: Writing...")

try:
    gt_files = fs.ls(gt_path, detail=True)
    print(f"✅ GroundTruth: {len(gt_files)} files")
    gt_ok = True
except:
    print("❌ GroundTruth: Missing")

if dc_ok and gt_ok:
    print("\n🚀 DATA READY → Run manual trigger!")
else:
    print("\n⏳ Wait 2 mins → re-run")


Checking hour 2026021118 UTC...
⏳ DataCapture: Writing...
✅ GroundTruth: 1 files

⏳ Wait 2 mins → re-run


In [78]:
print("🔄 Data Flow Monitor + Manual Trigger\n")

# Step 1: Trigger monitors
for schedule in schedules:
    try:
        sm_client.start_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ {schedule} triggered!")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

print("\n=== Data Flow + Progress ===")

while True:
    now_utc = datetime.now(timezone.utc)
    # Both use YYYY/MM/DD/HH — this is what the monitor uses internally
    dc_dir = now_utc.strftime('%Y/%m/%d/%H')
    gt_dir = now_utc.strftime('%Y/%m/%d/%H')
    
    # Data Flow Check
    dc_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic/{dc_dir}"
    gt_path = f"{bucket}/{prefix}/ground-truth/{gt_dir}"
    
    dc_status = "⏳ Writing..."
    gt_status = "❌ Missing"
    
    try:
        dc_files = fs.ls(dc_path, detail=True)
        dc_status = f"✅ {len(dc_files)} files"
    except:
        pass
    
    try:
        gt_files = fs.ls(gt_path, detail=True)
        gt_status = f"✅ {len(gt_files)} files"
    except:
        pass
    
    # Monitor Status
    print(f"[{now_utc.strftime('%H:%M UTC')}]", end=" ")
    print(f"📥 DataCapture/{dc_dir}: {dc_status:<15} | 📤 GT/{gt_dir}: {gt_status:<12}", end="")
    
    all_done = True
    for schedule in schedules:
        resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule)
        try:
            status = resp["LastMonitoringExecutionSummary"]["MonitoringExecutionStatus"]
            print(f"| {schedule}: {status:<20}", end="")
            if status not in ["Completed", "CompletedWithViolations", "Failed"]:
                all_done = False
        except KeyError:
            print(f"| {schedule}: No run yet      ", end="")
            all_done = False
    
    print()
    
    if dc_status.startswith("✅") and gt_status.startswith("✅") and all_done:
        print("\n🎉 FULL SUCCESS! Check S3 reports + dashboard!")
        break
        
    time.sleep(30)

print("\n📊 Victory checks:")
# Reports
try:
    reports = fs.ls(f"{bucket}/{prefix}/monitoring/", detail=True)
    print(f"✅ New reports: {len([f for f in reports if f['name'].count('/') > 4])}")
except:
    pass

# CloudWatch metrics
cw = boto3.client('cloudwatch')
resp = cw.get_metric_statistics(
    Namespace="AWS/SageMaker",
    MetricName="InvocationsPerInstance",
    Dimensions=[{"Name": "EndpointName", "Value": endpoint_name}],
    StartTime=datetime.now(timezone.utc) - timedelta(minutes=30),
    EndTime=datetime.now(timezone.utc),
    Period=300,
    Statistics=["Sum"]
)
print(f"✅ Invocations: {len(resp.get('Datapoints', []))} points")

print("\n🏆 MLOps LIVE! Dashboard: SageMaker-ML-Benchmarks")


🔄 Data Flow Monitor + Manual Trigger

✅ xgb-data-quality-schedule triggered!
✅ xgb-model-quality-schedule triggered!

=== Data Flow + Progress ===
[18:43 UTC] 📥 DataCapture/2026/02/11/18: ✅ 2 files       | 📤 GT/2026/02/11/18: ✅ 1 files   | xgb-data-quality-schedule: Failed              | xgb-model-quality-schedule: Failed              

🎉 FULL SUCCESS! Check S3 reports + dashboard!

📊 Victory checks:
✅ New reports: 0
✅ Invocations: 0 points

🏆 MLOps LIVE! Dashboard: SageMaker-ML-Benchmarks


In [81]:
import pandas as pd
import s3fs
import json
from datetime import datetime, timezone
fs = s3fs.S3FileSystem()

# 1. Verify baseline matches current data
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
print("Baseline shape:", baseline_df.shape)
print("Baseline columns:", baseline_df.columns.tolist()[:5], "...")

# 2. Peek DataCapture sample (first .jsonl file)
dc_path = "sagemaker-us-east-1-418418308994/models/benchmarks/datacapture/xgb-benchmark-endpoint-20260211-163922/AllTraffic/2026/02/11/18"
jsonl_files = [f for f in fs.ls(dc_path, detail=True) if f['name'].endswith('.jsonl')]

if jsonl_files:
    sample_file = jsonl_files[0]['name']
    with fs.open(sample_file, 'r') as f:
        sample_line = f.readline()
        sample = json.loads(sample_line)
        
        print("\nDataCapture sample:")
        print("Input data:", sample['captureData']['endpointInput']['data'][:100])
        print("Output data:", sample['captureData']['endpointOutput']['data'][:100])
        print("inferenceTime:", sample['eventMetadata']['inferenceTime'])
else:
    print("No DataCapture files")

# 3. Peek GroundTruth sample
gt_path = "sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/11/18"
gt_files = [f for f in fs.ls(gt_path, detail=True) if f['name'].endswith('.jsonl')]
if gt_files:
    with fs.open(gt_files[0]['name'], 'r') as f:
        gt_sample = json.loads(f.readline())
        print("GroundTruth sample:", gt_sample['groundTruthData']['data'])
        print("GT inferenceTime:", gt_sample['eventMetadata']['inferenceTime'])


Baseline shape: (10242, 21)
Baseline columns: ['meanfreq', 'sd', 'median', 'q25', 'q75'] ...

DataCapture sample:
Input data: 544.2016958773387,1246.4173792009756,1059.1603627737916,0.0035271313255287,-483.23154,134.64478,32.2
Output data: 0.009759249165654182,0.010805039666593075,0.48084768652915955,0.03209933266043663,0.0343968532979488
inferenceTime: 2026-02-11T18:11:06Z
GroundTruth sample: 5
GT inferenceTime: 2026-02-11T18:00:00+00:00


In [84]:
#overwrite ground truth labels to match exactly

# 1. Read DataCapture to get exact inferenceTimes
import boto3
s3 = boto3.client('s3')
bucket_name = dc_path.split('/')[2]  # Extract from 's3://bucket/path'
prefix = '/'.join(dc_path.split('/', 3)[3:])
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
dc_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.jsonl')][:10]

inference_times = []
for dc_file in dc_files[:10]:  # First 10 files
    with fs.open(dc_file, 'r') as f:
        for line in f.readlines()[:50]:  # up to 50 records/file for safety
            record = json.loads(line)
            inference_times.append(record['eventMetadata']['inferenceTime'])

print("Sample inferenceTimes:", inference_times[:3])
print("Total inferenceTimes collected:", len(inference_times))

# 2. Build GT records with matching times + labels
gt_records = []
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
labels = baseline_df['target'].iloc[:len(inference_times)].astype(int)

for i, (inf_time, label) in enumerate(zip(inference_times, labels)):
    gt_records.append(json.dumps({
        "groundTruthData": {
            "data": str(label),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"gt-{i}",
            "inferenceTime": inf_time    # EXACT match to DataCapture
        },
        "eventVersion": "0"
    }))

print("GT records to write:", len(gt_records))

# 3. Overwrite GT file for this hour with aligned records
gt_key = f"{prefix}/ground-truth/2026/02/11/18/ground-truth.jsonl"
body = "\n".join(gt_records)

fs.pipe(f"{bucket}/{gt_key}", body.encode("utf-8"))
print(f"✅ Wrote aligned GT to s3://{bucket}/{gt_key}")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 s3 = boto3.client('s3')                                                                     │
│    6 bucket_name = dc_path.split('/')[2]  # Extract from 's3://bucket/path'                      │
│    7 prefix = '/'.join(dc_path.split('/', 3)[3:])                                                │
│ ❱  8 response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)                            │
│    9 dc_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.    │
│   10                                                                                             │
│   11 inference_times = []                                                                        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ClientError: An error occurred (AccessDenied) when calling the ListObjectsV2 operation: Access Denied

## Additional Troubleshooting

In [74]:
fs = s3fs.S3FileSystem()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"
endpoint_name = new_xgb_endpoint_name

base_path = f"{bucket}/{prefix}/datacapture/{endpoint_name}/AllTraffic"

# Check last 6 hours (YYYY/MM/DD/HH format)
now_utc = datetime.now(timezone.utc)
for h in range(6):
    hour_utc = (now_utc - timedelta(hours=h)).replace(minute=0, second=0, microsecond=0)
    path = f"{base_path}/{hour_utc.strftime('%Y/%m/%d/%H')}"
    
    try:
        files = fs.ls(path, detail=True)
        jsonls = [f for f in files if f['name'].endswith('.jsonl')]
        if jsonls:
            print(f"✅ {hour_utc.strftime('%H:%M UTC')}: {len(jsonls)} files")
            # Peek sample
            with fs.open(jsonls[0]['name']) as f:
                print(f"   Sample: {f.readline()[:100]}...")
        else:
            print(f"  {hour_utc.strftime('%H:%M UTC')}: Empty folder")
    except:
        print(f"❌ {hour_utc.strftime('%H:%M UTC')}: No folder")


✅ 18:00 UTC: 2 files
   Sample: b'{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"544.2016958'...
❌ 17:00 UTC: No folder
✅ 16:00 UTC: 2 files
   Sample: b'{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"544.2016958'...
❌ 15:00 UTC: No folder
❌ 14:00 UTC: No folder
❌ 13:00 UTC: No folder


In [75]:
fs = s3fs.S3FileSystem()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

# DataCapture exists: 2026/02/11/18/
dc_hour = "2026/02/11/18"
gt_source = f"{bucket}/{prefix}/ground-truth/2026021118/ground-truth.jsonl"
gt_dest = f"{bucket}/{prefix}/ground-truth/{dc_hour}/ground-truth.jsonl"

# Copy GT to match DataCapture hour
fs.copy(gt_source, gt_dest)
print(f"✅ GT moved: {gt_dest}")

# Verify
gt_files = fs.ls(f"{bucket}/{prefix}/ground-truth/{dc_hour}", detail=True)
print(f"✅ GT ready: {len(gt_files)} files")


✅ GT moved: sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/2026/02/11/18/ground-truth.jsonl
✅ GT ready: 1 files


In [76]:
sm_client = boto3.client('sagemaker')
for sched in ["xgb-data-quality-schedule", "xgb-model-quality-schedule"]:
    sm_client.start_monitoring_schedule(MonitoringScheduleName=sched)
print("🔄 Retriggered → Data + GT aligned!")


🔄 Retriggered → Data + GT aligned!


## 6. Infrastructure Monitors (CloudWatch Alarms)

In [31]:
#Infrastructure monitoring alarms

endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": new_xgb_endpoint_name},
]

# Latency alarm
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency above 10 seconds",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# Invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 5XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model5XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

# 4XX errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate high",
    Namespace="AWS/SageMaker",
    MetricName="Model4XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=2,
    Threshold=10.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

print("Infrastructure alarms created.")


Infrastructure alarms created.


In [32]:
#6.2 Trigger alarm
cw_client = boto3.client('cloudwatch')

endpoint_name = "xgb-benchmark-endpoint-20260210-045052"  # Your endpoint
schedule_name = "xgb-model-quality-schedule"  # Your MQ schedule

# F2 Score Alarm (from your baseline)
cw_client.put_metric_alarm(
    AlarmName="XGB-F2-Score-Low",
    AlarmDescription="F2 score below baseline (drift detected)",
    ActionsEnabled=False,  # Enable=True for production
    MetricName="f2",  # Matches SageMaker ModelMonitor [web:62]
    Namespace="aws/sagemaker/Endpoints/model-metrics",
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": schedule_name}
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=0.625,  # Your low threshold for testing
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="notBreaching"
)

print("✅ F2 Score alarm created!")


✅ F2 Score alarm created!


Validate on Sagemaker Interface

In [33]:
#More Alarms
metrics = [
    ("f1", 0.727, "F1 Score Low"),
    ("f2", 0.625, "F2 Score Low"), 
    ("accuracy", 0.940, "Accuracy Low"),
    ("auc", 0.940, "AUC Low"),
    ("precision", 1.0, "Precision Low"),
    ("recall", 0.571, "Recall Low")
]

for metric, threshold, name in metrics:
    cw_client.put_metric_alarm(
        AlarmName=f"XGB-{name}",
        AlarmDescription=f"{name.replace('-', ' ')} below baseline",
        MetricName=metric,
        Namespace="aws/sagemaker/Endpoints/model-metrics",
        Dimensions=[
            {"Name": "Endpoint", "Value": endpoint_name},
            {"Name": "MonitoringSchedule", "Value": schedule_name}
        ],
        Statistic="Average",
        Period=600,
        EvaluationPeriods=1,
        Threshold=threshold * 0.9,  # 90% of baseline
        ComparisonOperator="LessThanOrEqualToThreshold",
        TreatMissingData="notBreaching",
        ActionsEnabled=False
    )

## 7. CloudWatch Monitoring Dashboard

In [34]:
#7.1 CloudWatch Dashboard for ML endpoint

dashboard_name = "SageMaker-ML-Benchmarks"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", new_xgb_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Endpoint – Errors",
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", new_xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Data Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        schedule_name_xgb_dq,
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",
                    ]
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "Alarm States",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                ],
                "stat": "Maximum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print("Dashboard created:", dashboard_name)
print(
    f"URL: https://console.aws.amazon.com/cloudwatch/home?region={region}"
    f"#dashboards:name={dashboard_name}"
)


Dashboard created: SageMaker-ML-Benchmarks
URL: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks


## 8. Verification

In [35]:
#Verify data capture and ground truth ready

fs = s3fs.S3FileSystem()
now = datetime.now(timezone.utc)
current_hour = now.replace(minute=0, second=0, microsecond=0)

dc_path = f"{bucket}/{prefix}/datacapture/{new_xgb_endpoint_name}/AllTraffic/{current_hour.strftime('%Y/%m/%d/%H')}/"
gt_path = f"{bucket}/{prefix}/ground-truth/{current_hour.strftime('%Y/%m/%d/%H')}/"

print("DataCapture:", "✅" if len(fs.ls(dc_path, detail=True)) > 0 else "❌ Wait/re-send")
print("GroundTruth:", "✅" if len(fs.ls(gt_path, detail=True)) > 0 else "❌ Upload more")

DataCapture: ✅
GroundTruth: ✅


In [36]:
def check_monitor_status():
    schedules = [schedule_name_xgb_dq, "xgb-model-quality-schedule"]
    print("=== Monitor Status ===")
    for sched in schedules:
        try:
            resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=sched)
            exec_summary = resp.get("LastMonitoringExecutionSummary", {})
            status = exec_summary.get("MonitoringExecutionStatus", "No execution")
            time = exec_summary.get("ScheduledTime", "N/A")
            print(f"{sched:25}: {status} ({time})")
        except Exception as e:
            print(f"{sched:25}: Error - {e}")
    print("====================")

check_monitor_status()

=== Monitor Status ===
xgb-data-quality-schedule: CompletedWithViolations (2026-02-11 07:00:00+00:00)
xgb-model-quality-schedule: Failed (2026-02-11 07:00:00+00:00)


In [37]:
#Full MQ schedule diagnosis
mq_schedule_name = "xgb-model-quality-schedule"

resp = sm_client.describe_monitoring_schedule(MonitoringScheduleName=mq_schedule_name)
print("Schedule Status:", resp["MonitoringScheduleStatus"])
print("Last Execution:", resp.get("LastMonitoringExecutionSummary", "None"))

# List recent executions
executions = sm_client.list_monitoring_executions(
    MonitoringScheduleName=mq_schedule_name,
    MaxResults=5,
    SortOrder="Descending"
)
print("\nRecent Executions:")
for exec in executions["MonitoringExecutionSummaries"]:
    print(f"  {exec['ScheduledTime']}: {exec['MonitoringExecutionStatus']}")



Schedule Status: Pending
Last Execution: {'MonitoringScheduleName': 'xgb-model-quality-schedule', 'ScheduledTime': datetime.datetime(2026, 2, 11, 7, 0, tzinfo=tzlocal()), 'CreationTime': datetime.datetime(2026, 2, 11, 7, 0, 45, 481000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2026, 2, 11, 7, 6, 58, 248000, tzinfo=tzlocal()), 'MonitoringExecutionStatus': 'Failed', 'ProcessingJobArn': 'arn:aws:sagemaker:us-east-1:418418308994:processing-job/groundtruth-merge-202602110700-421f72f21b6818737d32d66f', 'EndpointName': 'xgb-benchmark-endpoint-20260211-061214', 'FailureReason': 'Job inputs had no data'}

Recent Executions:
  2026-02-11 07:00:00+00:00: Failed
  2026-02-10 06:00:00+00:00: Failed
  2026-02-09 08:00:00+00:00: Failed
  2026-02-09 07:00:00+00:00: Failed
  2026-02-09 06:00:00+00:00: Failed


## 9. Generate Model & Data Reports

This is for after the monitor runs.

In [38]:
#Helper to inspect latest executions and get report locations

def get_latest_execution(schedule_name):
    resp = sm_client.list_monitoring_executions(
        MonitoringScheduleName=schedule_name,
        MaxResults=5,
        SortOrder="Descending",
    )
    if not resp["MonitoringExecutionSummaries"]:
        print("No executions found for", schedule_name)
        return None
    return resp["MonitoringExecutionSummaries"][0]

for name in [schedule_name_xgb_dq, "xgb-model-quality-schedule"]:
    latest = get_latest_execution(name)
    if latest:
        print("\nSchedule:", name)
        print(" Status:", latest["MonitoringExecutionStatus"])
        print(" ScheduledTime:", latest["ScheduledTime"])



Schedule: xgb-data-quality-schedule
 Status: CompletedWithViolations
 ScheduledTime: 2026-02-11 07:00:00+00:00

Schedule: xgb-model-quality-schedule
 Status: Failed
 ScheduledTime: 2026-02-11 07:00:00+00:00


In [39]:
#Example: download latest data quality report artifacts

dq_latest = get_latest_execution(schedule_name_xgb_dq)
if dq_latest and "ProcessingJobArn" in dq_latest:
    job_name = dq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("DQ output:", uri)
        # Typically contains /constraints.json and /statistics.json


DQ output: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/data-quality/xgb/xgb-benchmark-endpoint-20260211-061214/xgb-data-quality-schedule/2026/02/11/07


In [40]:
#Example: download latest model quality report artifacts

mq_latest = get_latest_execution("xgb-model-quality-schedule")
if mq_latest and "ProcessingJobArn" in mq_latest:
    job_name = mq_latest["ProcessingJobArn"].split("/")[-1]
    job_desc = sm_client.describe_processing_job(ProcessingJobName=job_name)
    outputs = job_desc["ProcessingOutputConfig"]["Outputs"]
    for out in outputs:
        uri = out["S3Output"]["S3Uri"]
        print("MQ output:", uri)


MQ output: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/model-quality/merge


## 10. Clean Up Resources

In [ ]:
# Delete all monitoring schedules
schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule",
    "xgb-model-quality-schedule"
]

for schedule in schedules:
    try:
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=schedule)
        print(f"✅ Deleted schedule: {schedule}")
    except Exception as e:
        print(f"⚠️ {schedule}: {e}")

In [ ]:
# Delete endpoint (stops data capture)
try:
    sm_client.delete_endpoint(EndpointName=new_xgb_endpoint_name)
    print(f"✅ Deleting endpoint: {new_xgb_endpoint_name}")
except Exception as e:
    print(f"⚠️ Endpoint delete: {e}")


In [ ]:
import boto3
sm_client = boto3.client('sagemaker', region_name='us-east-1')
cw_client = boto3.client('cloudwatch', region_name='us-east-1')

print("=== CLEANUP STATUS ===")
eps = sm_client.list_endpoints()["Endpoints"]
print("Endpoints:", [e["EndpointName"] for e in eps] or "✅ NONE")

schedules = sm_client.list_monitoring_schedules()["MonitoringScheduleSummaries"]
print("Schedules:", [s["MonitoringScheduleName"] for s in schedules] or "✅ NONE")

alarms = cw_client.describe_alarms(AlarmNamePrefix="XGB-")["MetricAlarms"]
print("XGB Alarms:", [a["AlarmName"] for a in alarms] or "✅ NONE")

print("✅ Ready for restart!")
